In [20]:
from scripts import lesion_diagnostics
import os
from pathlib import Path
import pandas as pd
import numpy as np
import nibabel as nib
import json
import pickle as pkl

import scripts.compile_run_metrics
import helpers.utils
import scripts.inference
from IPython.display import clear_output

from reload_recursive import reload_recursive


In [21]:
reload_recursive(scripts.compile_run_metrics)
#TODO Thi
from scripts.compile_run_metrics import EXPERIMENT_KEYS

reload_recursive(scripts.inference)
from scripts.inference import uncrop_predictions

reload_recursive(helpers.utils)
from helpers.utils import dice_score

import core.dataset
reload_recursive(core.dataset)
from core.dataset import Dataset
from core.experiment import Experiment

from scripts import lesion_diagnostics as lesion_dx
from scripts.compile_run_metrics import load_or_cache_run

In [ ]:
dataset_name = "roi_train2"
dataset = Dataset(dataset_name)

experiment_key, run_name = "stage3", "run2"
experiment_name = f"{EXPERIMENT_KEYS[experiment_key]}/{run_name}"

experiment_results = load_or_cache_run(dataset_name, experiment_name)
experiment = Experiment.from_run_dir(experiment_name, dataset)

expand_xy: int = experiment.preprocess_config.expand_xy
expand_z: int = experiment.preprocess_config.expand_z
images: tuple[str, ...] = experiment.preprocess_config.images

2026-03-26 14:06:11.540 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:185 - Loading cached: roi_train2_stage3_numcrops_bkd_constwt115_run2.pkl


In [25]:
import shelve

experiment.cases
with shelve.open("test") as db:
    db['experiment'] = experiment

In [26]:
with shelve.open("test") as db:
    loaded_experiment  = db['experiment']

In [29]:
from diskcache import Cache

cache = Cache(directory="/home/srs-9/Projects/prl_project/notebooks/tmp")
with Cache(cache.directory) as ref:
    ref.set("experiment", experiment)

In [ ]:
experiment.run_dir

In [ ]:
subject_dir: Path = Path("/media/smbshare/srs-9/prl_project/data/sub1011-20180911")

expand_xy: int = 20
expand_z: int = 2

data_root: Path = Path("/media/smbshare/srs-9/prl_project/data")
infer_output_dir = data_root
images: tuple[str, ...] = ["flair", "phase"]

def load_experiment(exp_name, run_name):
    path = cache_dir / f"{EXPERIMENT_KEYS[exp_name]}_{run_name}.pkl"
    with open(path, 'rb') as f:
        exp_data = pkl.load(f)
    return exp_data

experiment = load_experiment(experiment_key, run_name)
cases = pd.DataFrame(experiment['cases']).set_index(["subid", "lesion_index"])

with open("/home/srs-9/Projects/prl_project/training/roi_train2/datalist_flair.phase_xy20_z2.json", 'r') as f:
    datalist = json.load(f)
    
datalist_df = pd.DataFrame((datalist['training']+datalist['testing'])).set_index(["subid", "lesion_index"])

In [10]:
# Load lesion index once
subid = 1011
subject_dir = (data_root / datalist_df.loc[subid, "image"].iloc[0]).parent.parent
lesion_index_path = subject_dir / "lstai_lesion_index.nii.gz"
lesion_index = nib.load(str(lesion_index_path)).get_fdata().astype(np.int32)

# Parse bounding boxes
bbox_suffix = f"xy{expand_xy}_z{expand_z}"
bbox_file = subject_dir / f"lstai_bounding_boxes_{bbox_suffix}.txt"
bounding_boxes = lesion_diagnostics._parse_bounding_boxes(bbox_file)

# Build inference output filename pattern
image_basenames = sorted(images)
image_prefix = ".".join(image_basenames) + "_"
# going to need to get the appropriate datalist for suffxes and stuff
label_filename = f"{image_prefix}{bbox_suffix}.nii.gz"
subject_rel = subject_dir.relative_to(data_root)

In [11]:
# reload_recursive(scripts.inference)
# from scripts.inference import uncrop_predictions

# with open("/home/srs-9/Projects/prl_project/training/roi_train2/subjects.txt", 'r') as f:
#     subjects = [sub.strip() for sub in f.readlines()]

# for subid in subjects:
#     subid = int(subid)
#     subject_root = Path(cases.loc[subid, "image"].iloc[0]).parent.parent
#     infer_filepaths = {}
#     for index, row in cases.loc[subid].iterrows():
#         infer_filepaths[index] = row.loc["inference"]

#     uncrop_predictions(subject_root, expand_xy, expand_z, data_root, ["flair", "phase"], experiment_id, infer_filepaths=infer_filepaths)

In [13]:

subid = 1011
subject_root = Path(cases.loc[subid, "image"].iloc[0]).parent.parent
lesion_index_path = subject_dir / "lstai_lesion_index.nii.gz"
lesion_index = nib.load(str(lesion_index_path)).get_fdata().astype(np.int32)

# Parse bounding boxes
bbox_suffix = f"xy{expand_xy}_z{expand_z}"
bbox_file = subject_dir / f"lstai_bounding_boxes_{bbox_suffix}.txt"
bounding_boxes = lesion_diagnostics._parse_bounding_boxes(bbox_file)

# Build inference output filename pattern
image_basenames = sorted(images)
image_prefix = ".".join(image_basenames) + "_"
# going to need to get the appropriate datalist for suffxes and stuff
label_filename = f"{image_prefix}{bbox_suffix}.nii.gz"
subject_rel = subject_dir.relative_to(data_root)


prls = []
i = 0
for index, coords in bounding_boxes:
    # Load inference output for this ROI
    image_path = data_root / cases.loc[(subid,index), "image"]
    groundtruth_path = data_root / cases.loc[(subid,index), "label"]
    infer_path = data_root / cases.loc[(subid,index), "inference"]
    if not infer_path.exists():
        print(f"Inference output not found: {infer_path}")
        continue
    label_keys = ["truth", "infer"]
    lesion_dx = {"lesion_index": index, **{f"rim_voxels_{k}": None for k in label_keys}}
    for id, lab in zip(label_keys, [groundtruth_path, infer_path]):
        lab_data = nib.load(str(lab)).get_fdata().astype(np.uint8)

        # Crop lesion index with the same bounding box
        index_crop = lesion_diagnostics._crop_from_volume(lesion_index, coords)

        # any iron detection: rim voxels that overlap the central lesion's footprint
        has_iron = np.any((index_crop == index) & (lab_data == 2))

        if has_iron:
            # Connected component analysis to get reliable rim voxel count
            rim_count = lesion_diagnostics._count_rim_for_lesion(index_crop, lab_data, index)
            lesion_dx.update({f"rim_voxels_{id}": rim_count})

    if any([lesion_dx[f"rim_voxels_{k}"] for k in label_keys]):
    # if len(lesion_dx.keys()) > 1:  
        prls.append(lesion_dx)
prls

[{'lesion_index': 1, 'rim_voxels_truth': 272, 'rim_voxels_infer': 359},
 {'lesion_index': 2, 'rim_voxels_truth': None, 'rim_voxels_infer': 185},
 {'lesion_index': 3, 'rim_voxels_truth': 120, 'rim_voxels_infer': 139},
 {'lesion_index': 4, 'rim_voxels_truth': 345, 'rim_voxels_infer': 477},
 {'lesion_index': 5, 'rim_voxels_truth': None, 'rim_voxels_infer': 21},
 {'lesion_index': 6, 'rim_voxels_truth': 165, 'rim_voxels_infer': 210},
 {'lesion_index': 7, 'rim_voxels_truth': 189, 'rim_voxels_infer': 363},
 {'lesion_index': 8, 'rim_voxels_truth': None, 'rim_voxels_infer': 7},
 {'lesion_index': 10, 'rim_voxels_truth': None, 'rim_voxels_infer': 57},
 {'lesion_index': 12, 'rim_voxels_truth': 94, 'rim_voxels_infer': 49},
 {'lesion_index': 16, 'rim_voxels_truth': None, 'rim_voxels_infer': 49},
 {'lesion_index': 19, 'rim_voxels_truth': None, 'rim_voxels_infer': 5}]

In [14]:
import subprocess
from helpers.shell_interface import open_itksnap_workspace_cmd
index = 2
image_path = data_root / cases.loc[(subid,index), "image"]
groundtruth_path = data_root / cases.loc[(subid,index), "label"]
infer_path = data_root / cases.loc[(subid,index), "inference"]
label_keys = ["truth", "infer"]
labels = []
for id, lab in zip(label_keys, [groundtruth_path, infer_path]):
    labels.append(lab)
suffix = f"_xy{expand_xy}_z{expand_z}.nii.gz"
ims = ["flair", "phase"]
image_paths = []
for im in ims:
    name = f"{im}{suffix}"
    path = image_path.parent / name
    print(path)
    assert path.exists()
    image_paths.append(path)

subprocess.run(open_itksnap_workspace_cmd(image_paths, labels), shell=True)
clear_output()

- Convex Hull
- A sphere that encompasses the PRL voxels

In [ ]:
image_paths = [subject_root / "flair.nii.gz", subject_root / "phase.nii.gz"]
labels = [subject_root / f"prl_inference_{experiment_id}.nii.gz"]
assert labels[0].exists()
subprocess.run(open_itksnap_workspace_cmd(image_paths, labels), shell=True)
clear_output()

In [ ]:
import scripts.lesion_diagnostics as lesion_dx

subid = 1011
subject_root = Path(cases.loc[subid, "image"].iloc[0]).parent.parent
lesion_index_path = subject_dir / "lstai_lesion_index.nii.gz"
lesion_index = nib.load(str(lesion_index_path)).get_fdata().astype(np.int32)

# Parse bounding boxes
bbox_suffix = f"xy{expand_xy}_z{expand_z}"
bbox_file = subject_dir / f"lstai_bounding_boxes_{bbox_suffix}.txt"
bounding_boxes = lesion_diagnostics._parse_bounding_boxes(bbox_file)

# Build inference output filename pattern
image_basenames = sorted(images)
image_prefix = ".".join(image_basenames) + "_"
# going to need to get the appropriate datalist for suffxes and stuff
label_filename = f"{image_prefix}{bbox_suffix}.nii.gz"
subject_rel = subject_dir.relative_to(data_root)


prls = []
i = 0
for index, coords in bounding_boxes:
    # Load inference output for this ROI
    image_path = data_root / cases.loc[(subid,index), "image"]
    groundtruth_path = data_root / cases.loc[(subid,index), "label"]
    infer_path = data_root / cases.loc[(subid,index), "inference"]
    if not infer_path.exists():
        print(f"Inference output not found: {infer_path}")
        continue
    label_keys = ["truth", "infer"]
    lesion_dx = {"lesion_index": index, **{f"rim_voxels_{k}": None for k in label_keys}}
    for id, lab in zip(label_keys, [groundtruth_path, infer_path]):
        lab_data = nib.load(str(lab)).get_fdata().astype(np.uint8)

        # Crop lesion index with the same bounding box
        index_crop = lesion_diagnostics._crop_from_volume(lesion_index, coords)

        # any iron detection: rim voxels that overlap the central lesion's footprint
        has_iron = np.any((index_crop == index) & (lab_data == 2))

        if has_iron:
            rim

    if any([lesion_dx[f"rim_voxels_{k}"] for k in label_keys]):
    # if len(lesion_dx.keys()) > 1:  
        prls.append(lesion_dx)
prls

/media/smbshare/srs-9/prl_project/data/sub1011-20180911/32/flair_xy20_z2.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1011-20180911/32/phase_xy20_z2.nii.gz
